## Re-Ranking

![Re-Ranking Images](Images/Re-Ranking.png)

In [1]:
import bs4
from operator import itemgetter
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.load import dumps, loads

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# Indexing Phase

print("--- 1. Loading and Indexing ---")

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()
print(f"Loaded {len(blog_docs)} documents. Type: {type(blog_docs[0])}")

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50
)
splits = text_splitter.split_documents(blog_docs)
print(f"Created {len(splits)} splits. Type: {type(splits[0])}")

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()
print("Vectorstore and Retriever successfully created!\n")

--- 1. Loading and Indexing ---
Loaded 1 documents. Type: <class 'langchain_core.documents.base.Document'>
Created 50 splits. Type: <class 'langchain_core.documents.base.Document'>
Vectorstore and Retriever successfully created!



In [3]:
# RAG-Fusion Query Generation

print("--- 2. RAG-Fusion Query Generation ---")

template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_rag_fusion 
    | ChatOpenAI(temperature=0)
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

# Test the query generation under the hood
question = "What is task decomposition for LLM agents?"
generated_queries = generate_queries.invoke({"question": question})
print(f"Original Question: '{question}'")
print(f"Generated Queries: {generated_queries}")
print(f"Type of generated queries: {type(generated_queries)}\n")

--- 2. RAG-Fusion Query Generation ---
Original Question: 'What is task decomposition for LLM agents?'
Generated Queries: ['1. How do LLM agents use task decomposition in problem-solving?', '2. Benefits of task decomposition for LLM agents in artificial intelligence.', '3. Examples of task decomposition techniques used by LLM agents.', '4. Challenges of implementing task decomposition for LLM agents in machine learning.']
Type of generated queries: <class 'list'>



In [4]:
# Reciprocal Rank Fusion (RRF)

print("--- 3. Reciprocal Rank Fusion (RRF) ---")

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Takes multiple lists of ranked documents and fuses their scores using the RRF formula """
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    return [doc for doc, score in reranked_results]

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
fused_docs = retrieval_chain_rag_fusion.invoke({"question": question})

print(f"Retrieved and fused {len(fused_docs)} unique documents.")
print(f"Type of fused docs output list: {type(fused_docs)}")
print(f"Type of an individual fused doc: {type(fused_docs[0])}\n")

--- 3. Reciprocal Rank Fusion (RRF) ---
Retrieved and fused 6 unique documents.
Type of fused docs output list: <class 'list'>
Type of an individual fused doc: <class 'langchain_core.documents.base.Document'>



C:\Users\ashut\AppData\Local\Temp\ipykernel_21136\430312207.py:16: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (loads(doc), score)


In [5]:
# Final RAG Generation

print("--- 4. Final RAG Generation ---")

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

final_rag_chain = (
    {
        # We pipe the fused documents through our formatter so the LLM reads pure text
        "context": retrieval_chain_rag_fusion | format_docs, 
        "question": itemgetter("question")
    } 
    | prompt
    | llm
    | StrOutputParser()
)

final_answer = final_rag_chain.invoke({"question": question})
print(f"Type of final answer: {type(final_answer)}")
print(f"\nFinal Answer:\n{final_answer}")

--- 4. Final RAG Generation ---
Type of final answer: <class 'langchain_core.messages.base.TextAccessor'>

Final Answer:
Task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals. This allows the agent to efficiently handle complex tasks by dividing them into simpler steps. Task decomposition can be achieved through techniques like Chain of Thought (CoT) and Tree of Thoughts, which help the model think step by step and explore multiple reasoning possibilities at each step. By decomposing tasks, LLM agents can better interpret the model's thinking process and improve the quality of final results.


## Cohere Re-Rank

![Cohere Re-Rank](Images/Cohere_Re-Rank.png)

### (Note) : To run this, you will need a Cohere API key set in your environment variables as COHERE_API_KEY, but the LangChain wrapper handles the rest.

In [ ]:
# 1. Imports (Modernized)
import os
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

print("--- 1. Setting up Base Retriever (Stage 1) ---")
# We tell our fast Vector DB to fetch a wider net of 10 documents
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

print("--- 2. Initializing Cohere Reranker (Stage 2) ---")
# We set up Cohere to take those 10 docs, score them, and return only the top 3
compressor = CohereRerank(top_n=3) 

print("--- 3. Creating the Compression Pipeline ---")
# This wrapper links Stage 1 and Stage 2 together seamlessly
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=base_retriever
)

print("--- 4. Executing Two-Stage Retrieval ---")
question = "What is task decomposition for LLM agents?"

# .invoke() is the modern replacement for .get_relevant_documents()
compressed_docs = compression_retriever.invoke(question)

# Let's see what happened under the hood
print(f"Original question: '{question}'")
print(f"Number of documents returned after reranking: {len(compressed_docs)}")
print(f"Type of returned object: {type(compressed_docs)}")
print(f"Type of individual doc: {type(compressed_docs[0])}\n")

print("--- Top Reranked Document Preview ---")
# A cool feature of CohereRerank is that it injects its exact confidence score into the metadata!
print(f"Cohere Relevance Score: {compressed_docs[0].metadata.get('relevance_score')}")
print(f"Original Document ID/Source: {compressed_docs[0].metadata.get('source')}")
print(f"Content:\n{compressed_docs[0].page_content[:250]}...")

## Retrieval (CRAG)

`Deep Dive`

https://www.youtube.com/watch?v=E2shqsYwxck

`Notebooks`

https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag.ipynb

https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag_mistral.ipynb

## Retrieval (Self-RAG)
 
`Notebooks`

https://github.com/langchain-ai/langgraph/tree/main/examples/rag

https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_self_rag_mistral_nomic.ipynb

## Impact of long context  

`Deep dive`

https://www.youtube.com/watch?v=SsHUNfhF32s

`Slides`

https://docs.google.com/presentation/d/1mJUiPBdtf58NfuSEQ7pVSEQ2Oqmek7F1i4gBwR6JDss/edit#slide=id.g26c0cb8dc66_0_0